# 라스트레인 · 데이터 확장 (택시 대안 · 보류 버스 막차 · 주말 막차 · 좌표 · 실시간 도착)
**혁신기업경영 캡스톤디자인 II / Team 04 — v1.x 데이터 준비**

| 파트 | 채우는 것 | 기획서 항목 | 필요한 키 (Colab 🔑 보안 비밀 이름) |
|---|---|---|---|
| A | 출발지·도착지·역/정류장 **좌표** | 16 GPS 위치 안내 준비 | `KAKAO_REST_API_KEY` |
| B | 출발지→도착지 **택시 예상 요금·시간** | 13 대안 비교 | `KAKAO_REST_API_KEY` (+ 길찾기 API 사용 권한) |
| C | 판정 보류 버스 **5511 · 074 막차** | 데이터 공백 해소 | `DATA_GO_KR_KEY` |
| D | R07~R11 **토·일 지하철 막차** | 데이터 공백 해소 | `ODsay_API` (+ Server IP 등록) |
| E | 지하철 **실시간 도착** 시연 | 16 실시간 API 가능성 검증 | `SEOUL_SUBWAY_KEY` |
| F | 결과 묶어서 `lastrain_extra.json` 저장 | — | — |

**파트는 서로 독립**이다. 키가 없는 파트는 자동으로 건너뛴다. 있는 키부터 돌리면 된다.

### 원칙 (앱과 동일)
- API가 준 값만 쓴다. 못 받은 값은 비워 두고 **지어내지 않는다.**
- 이름 검색으로 찾은 장소는 **표로 출력해 사람이 눈으로 확인**한다(동명 장소 오매칭 방지).
- 키는 코드에 쓰지 않는다. 전부 Colab 보안 비밀에서 읽는다.

## 0. 키 발급처 (무료)
- **`KAKAO_REST_API_KEY`** — Kakao Developers → 내 애플리케이션 → 앱 키의 *REST API 키*.
  - 파트 A(장소 검색)는 키만 있으면 된다.
  - 파트 B(카카오모빌리티 자동차 길찾기)는 **사용 권한 신청·승인**이 필요할 수 있다(데브톡에 권한 요청 글 다수). 401/403이 나오면 승인 대기 상태.
- **`DATA_GO_KR_KEY`** — 공공데이터포털에서 아래 두 서비스 **활용신청** 후 마이페이지의 *일반 인증키(Decoding)*.
  - `서울특별시_노선정보조회 서비스` (5511용)
  - `경기도_버스노선 조회` (074용)
- **`ODsay_API`** — ODsay LAB. 앱 설정의 **Server IP에 아래 1번 셀이 찍는 IP를 등록**. 런타임 바뀌면 IP도 바뀜.
- **`SEOUL_SUBWAY_KEY`** — 서울 열린데이터광장 → *실시간 지하철 인증키* (일반 인증키와 별도).

> 보안 비밀 등록 후 각 항목의 **노트북 액세스** 토글을 켜야 읽힌다.

## 1. 설치 · 공통

In [ ]:
!pip -q install requests pandas

import requests, json, re, time, datetime
import pandas as pd
pd.set_option("display.max_colwidth", 50); pd.set_option("display.width", 220)

LOG = []
def log(step, status, detail=""):
    LOG.append({"단계": step, "상태": status, "내용": str(detail)[:300]})
    print(f"[{status}] {step} :: {str(detail)[:180]}")

try:
    from google.colab import userdata
    def getkey(n):
        try: return (userdata.get(n) or "").strip()
        except Exception: return ""
except ImportError:
    import os
    def getkey(n): return (os.environ.get(n) or "").strip()

KAKAO   = getkey("KAKAO_REST_API_KEY")
DATAGO  = getkey("DATA_GO_KR_KEY")
ODSAY   = getkey("ODsay_API")
SEOULRT = getkey("SEOUL_SUBWAY_KEY")
for n, v in [("KAKAO_REST_API_KEY", KAKAO), ("DATA_GO_KR_KEY", DATAGO), ("ODsay_API", ODSAY), ("SEOUL_SUBWAY_KEY", SEOULRT)]:
    print(f"{n:20} {'있음 (' + str(len(v)) + '자)' if v else '없음 → 해당 파트 건너뜀'}")

try:
    print("\n이 런타임 IP (ODsay Server IP에 등록):", requests.get("https://api.ipify.org", timeout=5).text)
except Exception as e:
    print("IP 확인 실패:", e)

RUN_AT = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime("%Y-%m-%d %H:%M KST")
print("실행 시각:", RUN_AT)

## 2. 조원 조사 경로 11개 불러오기
깃허브 저장소의 `data/routes.json`(조원 실측 원본)을 그대로 읽는다.

In [ ]:
RAW = "https://raw.githubusercontent.com/rnjswldnjs/lastrain/main/data/routes.json"
ROUTES = requests.get(RAW, timeout=10).json()["routes"]
rides = lambda r: [l for l in r["legs"] if l["mode"] != "walk"]
display(pd.DataFrame([{"경로": r["id"], "출발": r["from"], "도착": r["to"],
                       "탈것": " → ".join(l["name"] for l in rides(r))} for r in ROUTES]))

## A. 좌표 — 카카오 로컬 키워드 검색
- 역은 `"강남역 신분당선"`처럼 노선을 붙여 검색한다.
- 출발지·정류장은 **바로 앞에서 찾은 지점 기준 반경 3km 안**에서 가까운 순으로 찾는다(동명 지점 오매칭 방지).
- 결과 표의 `확인` 칸이 `⚠`이면 검색어 핵심어가 장소명에 없다는 뜻 → **눈으로 확인 후 아래 `OVERRIDE`에 고쳐 넣기.**

In [ ]:
KAKAO_H = {"Authorization": f"KakaoAK {KAKAO}"}

def kakao_kw(query, near=None, radius=3000):
    if not KAKAO: return []
    p = {"query": query, "size": 5}
    if near: p.update(x=near["x"], y=near["y"], radius=radius, sort="distance")
    try:
        r = requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", headers=KAKAO_H, params=p, timeout=10)
    except Exception as e:
        log("카카오 장소검색", "실패", e); return []
    if r.status_code != 200:
        log("카카오 장소검색", "실패", f"{r.status_code} {r.text[:150]}"); return []
    return r.json().get("documents", [])

def station_q(name, line):
    base = name if name.endswith("역") else name + "역"
    return f"{base} {line.replace(' 급행','')}"

# 자동 검색이 틀렸을 때 사람이 고치는 칸: {(경로, 역할): "검색어"}
OVERRIDE = {
    # ("R01", "출발"): "스타벅스 강남몬테소리점",   ← 예시
}

def pick(rid, role, query, near=None, core=None):
    q = OVERRIDE.get((rid, role), query)
    docs = kakao_kw(q, near) or (kakao_kw(q) if near else [])
    if not docs:
        return {"경로": rid, "역할": role, "검색어": q, "장소명": None, "x": None, "y": None, "주소": None, "확인": "❌ 없음"}
    d = docs[0]
    token = (core or q.split()[0]).replace("역", "")
    return {"경로": rid, "역할": role, "검색어": q, "장소명": d["place_name"], "x": d["x"], "y": d["y"],
            "주소": d.get("road_address_name") or d.get("address_name"),
            "분류": d.get("category_name", "")[-20:], "확인": "OK" if token[:2] in d["place_name"] else "⚠"}

COORD = []
if KAKAO:
    for r in ROUTES:
        rid, legs = r["id"], r["legs"]
        R = rides(r)
        first = R[0]
        # 1) 첫 탑승역 (기준점)
        b0 = pick(rid, "탑승1", station_q(first["from"], first["name"]) if first["mode"]=="sub" else first["from"]+" 정류장",
                  core=first["from"])
        COORD.append(b0)
        anchor = b0 if b0["x"] else None
        # 2) 출발지 (첫 구간이 도보면 별도 장소)
        if legs[0]["mode"] == "walk":
            COORD.append(pick(rid, "출발", r["from"], near=anchor, core=r["from"]))
        # 3) 각 탈것의 하차 지점 + 다음 탑승 지점
        prev = anchor
        for i, l in enumerate(R):
            if i > 0:
                q = station_q(l["from"], l["name"]) if l["mode"]=="sub" else l["from"]
                p = pick(rid, f"탑승{i+1}", q, near=prev, core=l["from"]); COORD.append(p); prev = p if p["x"] else prev
            q = station_q(l["to"], l["name"]) if l["mode"]=="sub" else l["to"]
            a = pick(rid, f"하차{i+1}", q, near=None if l["mode"]=="sub" else prev, core=l["to"])
            COORD.append(a); prev = a if a["x"] else prev
        # 4) 도착지 (역 이름)
        COORD.append(pick(rid, "도착", r["to"], near=prev, core=r["to"]))
        time.sleep(.15)
    CDF = pd.DataFrame(COORD)
    display(CDF)
    print("⚠·❌ 개수:", (CDF["확인"] != "OK").sum(), "→ 표에서 확인 후 OVERRIDE에 고치고 이 셀 다시 실행")
else:
    CDF = pd.DataFrame(); log("파트 A", "건너뜀", "KAKAO_REST_API_KEY 없음")

## B. 택시 예상 요금·시간 — 카카오모빌리티 자동차 길찾기
- 응답의 `summary.fare.taxi`(택시 예상 요금, 원)·`fare.toll`(통행료)·`distance`(m)·`duration`(초)을 쓴다.
- **호출 시점의 교통 상황 기준**이다. 심야(23~01시) 값을 원하면 그 시간대에 다시 돌리거나, 아래 *미래 시각 길찾기* 셀을 시도.
- **심야 할증이 요금에 반영되는지는 문서에서 확인되지 않았다.** 앱·기획서에는 "카카오모빌리티 예상 요금(호출 시각 기준)"으로만 표기할 것.

In [ ]:
def point(rid, role):
    if CDF.empty: return None
    m = CDF[(CDF["경로"]==rid) & (CDF["역할"]==role) & CDF["x"].notna()]
    return None if m.empty else m.iloc[0]

def car(o, d, future=None):
    url = "https://apis-navi.kakaomobility.com/v1/" + ("future/directions" if future else "directions")
    p = {"origin": f"{o['x']},{o['y']}", "destination": f"{d['x']},{d['y']}", "priority": "RECOMMEND"}
    if future: p["departure_time"] = future
    r = requests.get(url, headers=KAKAO_H, params=p, timeout=15)
    if r.status_code != 200:
        return {"오류": f"{r.status_code} {r.text[:120]}"}
    rt = (r.json().get("routes") or [{}])[0]
    if rt.get("result_code") != 0:
        return {"오류": f"result_code {rt.get('result_code')} {rt.get('result_msg')}"}
    s = rt["summary"]
    return {"거리km": round(s["distance"]/1000, 1), "시간분": round(s["duration"]/60),
            "택시요금": s.get("fare", {}).get("taxi"), "통행료": s.get("fare", {}).get("toll")}

TAXI = []
if KAKAO and not CDF.empty:
    for r in ROUTES:
        o = point(r["id"], "출발") if r["legs"][0]["mode"]=="walk" else point(r["id"], "탑승1")
        d = point(r["id"], "도착")
        if o is None or d is None:
            TAXI.append({"경로": r["id"], "오류": "좌표 없음"}); continue
        TAXI.append({"경로": r["id"], "출발": r["from"], "도착": r["to"], **car(o, d), "기준": RUN_AT})
        time.sleep(.2)
    TDF = pd.DataFrame(TAXI); display(TDF)
    if "오류" in TDF and TDF["오류"].astype(str).str.contains("401|403").any():
        print("→ 401/403: 카카오모빌리티 길찾기 사용 권한 승인 전일 가능성. 승인 후 다시 실행.")
else:
    TDF = pd.DataFrame(); log("파트 B", "건너뜀", "키 또는 좌표 없음")

In [ ]:
# (선택) 미래 시각 길찾기 — 다음 평일 23:30 출발 기준. 엔드포인트가 안 되면 오류만 찍고 넘어간다.
FUTURE = None
if KAKAO and not CDF.empty:
    nxt = datetime.datetime.now() + datetime.timedelta(days=1)
    while nxt.weekday() >= 5: nxt += datetime.timedelta(days=1)
    FUTURE = nxt.strftime("%Y%m%d") + "2330"
    rows = []
    for r in ROUTES[:3]:   # 먼저 3개만 시험
        o = point(r["id"], "출발") if r["legs"][0]["mode"]=="walk" else point(r["id"], "탑승1")
        d = point(r["id"], "도착")
        if o is not None and d is not None:
            rows.append({"경로": r["id"], "출발시각": FUTURE, **car(o, d, future=FUTURE)})
    display(pd.DataFrame(rows))
    print("오류 없이 나오면 ROUTES[:3]를 ROUTES로 바꿔 전체 실행 → 심야 기준 값으로 쓸 수 있음")

## C. 판정 보류 버스 막차 — 5511(서울) · 074(경기)
앱에서 **R10(5511)·R09(074)** 가 막차 미확인이라 판정 보류 중이다.

- 노선 API가 주는 막차는 보통 **기점 출발 기준**이다. 우리 정류장(농생대·운정중앙역) 도착 시각이 아니다.
- 그래서 정류장별 시간 필드가 있으면 그걸 쓰고, 없으면 **"기점 막차"로만 기록**한다(앱에도 '추정' 표시).
- 응답 구조가 문서와 다르면 원문 일부를 찍으니 그대로 보내줄 것.

In [ ]:
def get_json(url, params):
    try:
        r = requests.get(url, params=params, timeout=15)
    except Exception as e:
        return None, f"요청 실패 {e}"
    try:
        return r.json(), None
    except Exception:
        return None, f"JSON 아님 ({r.status_code}): {r.text[:250]}"

BUS = []
if DATAGO:
    # ── 5511 (서울특별시_노선정보조회)
    j, err = get_json("http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList",
                      {"serviceKey": DATAGO, "strSrch": "5511", "resultType": "json"})
    items = ((j or {}).get("msgBody") or {}).get("itemList") or []
    print("5511 노선목록 응답:", err or (j or {}).get("msgHeader"))
    route = next((it for it in items if str(it.get("busRouteNm")) == "5511"), None)
    if route:
        print("노선 필드:", list(route.keys()))
        rec = {"노선": "5511", "routeId": route.get("busRouteId"), "기점": route.get("stStationNm"), "종점": route.get("edStationNm"),
               "기점첫차": route.get("firstBusTm"), "기점막차": route.get("lastBusTm"), "정류장": "농생대", "정류장막차": None, "비고": ""}
        j2, err2 = get_json("http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute",
                            {"serviceKey": DATAGO, "busRouteId": route.get("busRouteId"), "resultType": "json"})
        st = ((j2 or {}).get("msgBody") or {}).get("itemList") or []
        if st: print("정류장 필드:", list(st[0].keys()))
        hits = [s for s in st if "농생대" in str(s.get("stationNm"))]
        for h in hits:
            print("농생대 매칭:", {k: h.get(k) for k in h if any(t in k.lower() for t in ("nm", "seq", "dir", "tm", "time"))})
        tfield = next((k for k in (hits[0].keys() if hits else []) if k.lower() in ("lasttm", "lastbustm", "lasttime")), None)
        if hits and tfield:
            rec["정류장막차"] = hits[0][tfield]; rec["비고"] = f"정류장 필드 {tfield}"
        else:
            rec["비고"] = "정류장별 막차 필드 없음 → 기점 막차만 확보 (추정 표시 필요)"
        BUS.append(rec)
    else:
        log("5511", "실패", err or f"노선 못 찾음 / 응답 일부: {str(j)[:200]}")

    # ── 074 (경기도_버스노선 조회) — 파주 노선만
    j, err = get_json("https://apis.data.go.kr/6410000/busrouteservice/v2/getBusRouteListv2",
                      {"serviceKey": DATAGO, "keyword": "074", "format": "json"})
    print("\n074 노선목록 응답:", err or "OK")
    body = (j or {}).get("response", {}).get("msgBody", {}) if j else {}
    lst = body.get("busRouteList") or []
    lst = lst if isinstance(lst, list) else [lst]
    cands = [x for x in lst if str(x.get("routeName")) == "074"]
    display(pd.DataFrame(cands)[[c for c in ("routeId","routeName","regionName","routeTypeName","stStationName","edStationName") if cands and c in cands[0]]] if cands else "후보 없음 — 응답 일부: " + str(j)[:300])
    paju = next((x for x in cands if "파주" in str(x.get("regionName"))), None)
    if paju:
        j3, err3 = get_json("https://apis.data.go.kr/6410000/busrouteservice/v2/getBusRouteInfoItemv2",
                            {"serviceKey": DATAGO, "routeId": paju["routeId"], "format": "json"})
        info = (j3 or {}).get("response", {}).get("msgBody", {}).get("busRouteInfoItem") if j3 else None
        print("074 상세 필드:", list(info.keys()) if info else (err3 or str(j3)[:300]))
        if info:
            BUS.append({"노선": "074", "routeId": paju["routeId"], "기점": info.get("startStationName"), "종점": info.get("endStationName"),
                        "기점첫차": info.get("upFirstTime"), "기점막차": info.get("upLastTime"),
                        "역방향막차": info.get("downLastTime"), "정류장": "운정중앙역", "정류장막차": None,
                        "비고": "경기 API는 기점(상·하행) 첫·막차만 제공 → 운정중앙역 통과 시각 아님, 추정 표시 필요"})
    else:
        log("074", "확인필요", "파주 지역 074 후보 없음 — 후보 표를 보고 routeId를 직접 지정")
    BDF = pd.DataFrame(BUS); display(BDF)
else:
    BDF = pd.DataFrame(); log("파트 C", "건너뜀", "DATA_GO_KR_KEY 없음")

## D. 토·일 지하철 막차 — ODsay `subwayTimeTable`
지난번 노트북과 같은 방식: **조원 평일 막차를 정답지로 방향을 확정**한 뒤 토(dayType=2)·일(3) 막차를 받는다.
R01~R06은 기존 조사값과 **대조용**, R07~R11은 **새로 채우는 값**.

> 먼저 1번 셀이 찍은 IP를 ODsay 콘솔 Server IP에 등록하고 1분 기다릴 것.

In [ ]:
ODSAY_BASE = "https://api.odsay.com/v1/api/"
def odsay(endpoint, quiet=True, **params):
    if not ODSAY: return None
    try:
        data = requests.get(ODSAY_BASE + endpoint, params={**params, "apiKey": ODSAY}, timeout=10).json()
    except Exception as e:
        log(f"ODsay/{endpoint}", "실패", e); return None
    if "error" in data:
        if not quiet: log(f"ODsay/{endpoint}", "실패", data["error"])
        return None
    return data

def to_min(t): h, m = map(int, t.split(":")); return (h + 24 if h < 4 else h) * 60 + m

def extract_last(tt):
    out = {}
    for way, body in ((tt or {}).get("result", {}).get("OrdList", {}) or {}).items():
        best = None
        for r in (body or {}).get("time", []):
            try: h = int(r["Idx"])
            except Exception: continue
            hh = h + 24 if h < 4 else h
            for mm, dest in re.findall(r'(\d{1,2})\(([^)]*)\)', str(r.get("list", ""))):
                k = hh * 60 + int(mm)
                if best is None or k > best[0]: best = (k, h % 24, int(mm), dest)
        if best: out[way] = (f"{best[1]:02d}:{best[2]:02d}", best[3])
    return out

CACHE = {}
def find_station(name, line):
    if (name, line) in CACHE: return CACHE[(name, line)]
    c = ((odsay("searchStation", stationName=name, stationClass="2") or {}).get("result", {}) or {}).get("station", [])
    tgt = line.replace("GTX-A", "GTX").replace("급행", "").strip()
    best = next((x for x in c if tgt in str(x.get("laneName", ""))), c[0] if c else None)
    if best and tgt not in str(best.get("laneName", "")):
        log("역검색", "주의", f"{name}/{line} 노선 불일치 → {[(x['stationName'], x.get('laneName')) for x in c][:4]}")
    CACHE[(name, line)] = best; return best

WEEK = []
if ODSAY and odsay("searchStation", quiet=False, stationName="신촌", stationClass="2"):
    for r in ROUTES:
        for l in r["legs"]:
            if l["mode"] != "sub": continue
            known, known_we = l["last"]["weekday"], l["last"].get("weekend")
            s = find_station(l["from"], l["name"])
            if not s: WEEK.append({"경로": r["id"], "역": l["from"], "노선": l["name"], "상태": "역 못 찾음"}); continue
            cand = {}
            for way in (1, 2):
                for k, v in extract_last(odsay("subwayTimeTable", stationID=s["stationID"], wayCode=way, dayType=1)).items():
                    cand[(way, k)] = v
                time.sleep(.25)
            if not cand: WEEK.append({"경로": r["id"], "역": l["from"], "노선": l["name"], "상태": "시간표 없음"}); continue
            (way, wk), (t_wd, dest) = min(cand.items(), key=lambda kv: abs(to_min(kv[1][0]) - to_min(known)))
            sat = extract_last(odsay("subwayTimeTable", stationID=s["stationID"], wayCode=way, dayType=2)).get(wk)
            sun = extract_last(odsay("subwayTimeTable", stationID=s["stationID"], wayCode=way, dayType=3)).get(wk)
            diff = to_min(t_wd) - to_min(known)
            WEEK.append({"경로": r["id"], "역": l["from"], "노선": l["name"], "평일(조사)": known, "평일(API)": t_wd,
                         "차이(분)": diff, "방면": dest, "토": sat[0] if sat else None, "일": sun[0] if sun else None,
                         "토·일(조사)": known_we, "상태": "확인필요" if abs(diff) > 10 else "OK"})
            time.sleep(.25)
    WDF = pd.DataFrame(WEEK); display(WDF)
else:
    WDF = pd.DataFrame(); log("파트 D", "건너뜀", "ODsay 키 없음 또는 인증 실패(IP 등록 확인)")

## E. 지하철 실시간 도착 — 시연용 (서울 열린데이터광장)
기획서의 *"실시간 도착 API로 대체 가능"* 을 **실제로 받아온 증거**로 쓰는 셀. 결과 표를 캡처해서 기획서에 넣는다.
- 심야(막차 이후)엔 결과가 적거나 없을 수 있다 → 낮·저녁에 한 번 돌려 두기.
- 웹페이지(https)에서 이 API를 바로 부를 수 있는지는 별도 확인 필요. 이번엔 **Colab 시연까지만.**

In [ ]:
RT = []
if SEOULRT:
    for stn in ["강남", "사당", "홍대입구"]:
        j, err = get_json(f"http://swopenAPI.seoul.go.kr/api/subway/{SEOULRT}/json/realtimeStationArrival/0/8/{stn}", {})
        lst = (j or {}).get("realtimeArrivalList") or []
        if not lst: log("실시간 도착", "없음", err or str(j)[:200]); continue
        for a in lst:
            RT.append({"역": stn, "방면": a.get("trainLineNm"), "도착 메시지": a.get("arvlMsg2"),
                       "현재 위치": a.get("arvlMsg3"), "남은 초": a.get("barvlDt"), "수신": a.get("recptnDt")})
        time.sleep(.2)
    RDF = pd.DataFrame(RT); display(RDF)
else:
    RDF = pd.DataFrame(); log("파트 E", "건너뜀", "SEOUL_SUBWAY_KEY 없음")

## F. 저장 — 앱에 넣을 `lastrain_extra.json` + 확인용 CSV

In [ ]:
def recs(df): return [] if df is None or df.empty else json.loads(df.to_json(orient="records", force_ascii=False))

EXTRA = {
    "_meta": {"generated": RUN_AT,
              "sources": {"coords": "카카오 로컬 키워드 검색", "taxi": "카카오모빌리티 자동차 길찾기 summary.fare (호출 시각 기준)",
                          "bus_last": "서울특별시_노선정보조회 / 경기도_버스노선 조회 (기점 기준)", "weekend_last": "ODsay subwayTimeTable dayType 2·3",
                          "realtime": "서울 열린데이터광장 실시간 지하철 도착정보 (시연)"},
              "rule": "API가 준 값만 기록. 못 받은 값은 null. 사람 확인 전 좌표(확인≠OK)는 앱에 쓰지 않음."},
    "coords": recs(CDF), "taxi": recs(TDF), "bus_last": recs(BDF), "weekend_last": recs(WDF), "realtime_demo": recs(RDF),
}
with open("lastrain_extra.json", "w", encoding="utf-8") as f: json.dump(EXTRA, f, ensure_ascii=False, indent=1)
for name, df in [("coords", CDF), ("taxi", TDF), ("bus_last", BDF), ("weekend_last", WDF), ("realtime_demo", RDF)]:
    if not df.empty: df.to_csv(f"lastrain_{name}.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(LOG).to_csv("lastrain_extra_log.csv", index=False, encoding="utf-8-sig")

print("파트별 행 수:", {k: len(v) for k, v in EXTRA.items() if k != "_meta"})
try:
    from google.colab import files
    files.download("lastrain_extra.json"); files.download("lastrain_extra_log.csv")
except Exception as e:
    print("다운로드 건너뜀:", e)

---
### 돌린 뒤 보내줄 것
1. **`lastrain_extra.json`** (이것만 있으면 앱에 넣을 수 있음)
2. **`lastrain_extra_log.csv`** — 실패·주의 기록
3. 파트 A 표에서 **⚠·❌ 행** — OVERRIDE로 고쳤으면 고친 뒤 결과
4. 파트 C·E가 *"JSON 아님"* 이나 필드 목록만 찍었다면 **그 출력 그대로** (응답 구조 맞춰 파서 고침)

### 미리 알아둘 것
- **B 택시**: 401/403이면 카카오모빌리티 길찾기 권한 승인 전. 요금은 호출 시각 교통 기준이고 심야 할증 반영 여부는 확인 안 됨 → 앱엔 "예상 요금"으로만 표기.
- **C 버스**: 기점 막차라 우리 정류장 통과 시각과 다름 → 앱에서 판정 보류를 풀더라도 "추정" 표시.
- **D 지하철**: GTX-A·김포골드라인·신림선은 ODsay 수록 여부 불확실. 없으면 그 구간만 수기 확인.
- **E 실시간**: 시연용. 앱에는 넣지 않고 기획서 캡처로 사용.